# 12 — Ax Bayesian optimization for damped_sw α (CTL-03)

**Notebook:** SOO — single-objective profit.

**Decision:** ADR 0060 / CTL-03=B — tune the demand fractile `alpha` for a ladder controller by maximizing closed-loop **episode profit** (SIM-01=B), using [Ax](https://ax.dev/) instead of a fixed grid.

Each Ax trial evaluates one candidate **(α, ρ)** pair on **K stochastic demand realizations** (distinct `root_seed`s). We report `(mean, sem)` to Ax so observation noise is explicit.

Scoring uses `evaluate_alpha_episode_outcomes(TUNE_ARM, ...)` in `sim/alpha_tune.py` — **Rust-first** when `blueberries_voi._core` is built (including calendar demand via typed `_core.DemandProfile`).

With calendar demand on, protection targets use **Monte Carlo** sums of heterogeneous daily NB draws μ(day+k) (CAL-B4 / ADR 0134), not flat μ=30.

**Defaults are smoke-sized.** Set `FULL_RUN = True` for longer episodes and more BO trials.

**Policy:** damped survival-weighted (`damped_sw`) ordering only — damped survival-weighted ordering only.

**Calendar demand + BO:** use `FULL_RUN = True` (`n_burn=n_score=28`, four weeks) so burn-in covers the MWF order cadence. Smoke `n_burn=2` makes Ax objectives extremely noisy and can look flat even when the Rust kernel is healthy.

## Setup

From the repo root:

```bash
uv sync --extra notebooks --extra viz --extra rust --extra modal --extra data
uv run maturin develop --release --manifest-path crates/voi_py/Cargo.toml
modal token new   # once, if using Modal
uv run jupyter lab
```

**Concurrency:** Ax batch parallelism × `K_BO_SEEDS` flat Modal shards (see BO cell).

Set `RELOAD_AX = True` and `EXTRA_AX_TRIALS = N` to continue a saved Ax client without re-running completed trials.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from tqdm.auto import tqdm

from blueberries_voi.backend import rust_available, rust_core
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

from blueberries_voi.experiments.damped_sw_soo import (
    DampedSwSooBudgets,
    aggregate_soo_shards,
    build_soo_jobs,
    run_soo_shard,
)
from blueberries_voi.sim.alpha_tune import evaluate_alpha_episode_outcomes
from blueberries_voi.sim.bakeoff_damped_sw import protection_demand_quantile
from blueberries_voi.model import ModelParams
from blueberries_voi.model.demand_profile import load_demand_profile
from blueberries_voi.sim.order_schedule import DEFAULT_ORDER_SCHEDULE
from blueberries_voi.sim.profit import ProfitCosts
from blueberries_voi.sim.shipments import default_shipments, smoke_cool_shipments

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "blueberries_voi").is_dir():
    REPO_ROOT = REPO_ROOT.parent

POLICY = "damped_sw"
TUNE_ARM = "sw"
USE_MODAL = True  # flat (trial × seed) shards on Modal

FULL_RUN = True
ALPHA_BOUNDS = (0.1, 0.9999)

# 2026-08-30: retune for the widened rho range (was (0.5, 1.0)) and a deeper
# search, split into 4 batches of 12 trials each (48 total, was one 24-trial
# run) so a manual run can be stopped after any batch — see
# .team/plans/2026-08-29-gsin-belief-accuracy-blog-post.md §2.1-2.2.
# ax_parallelism=4 per Oliver (was 12 in an earlier draft of this cell).
if FULL_RUN:
    N_BURN, N_SCORE = 28, 28
    K_BO_SEEDS = 6
    TRIALS_PER_BATCH = 12
    N_BATCHES = 4
    TOTAL_AX_TRIALS = TRIALS_PER_BATCH * N_BATCHES  # 48
    EXTRA_AX_TRIALS = 0
    AX_PARALLELISM = 4
    MODAL_CONCURRENCY = 100
else:
    N_BURN, N_SCORE = 2, 5
    K_BO_SEEDS = 4
    TRIALS_PER_BATCH = 6
    N_BATCHES = 1
    TOTAL_AX_TRIALS = TRIALS_PER_BATCH * N_BATCHES
    EXTRA_AX_TRIALS = 0
    AX_PARALLELISM = 2
    MODAL_CONCURRENCY = 8

AX_JSON = REPO_ROOT / "outputs" / "damped_sw_alpha_bo_ax_client.json"
RELOAD_AX = False
_MODAL_DEMAND_PROFILE = "/data/freshnet/demand_profile.json"

RNG = np.random.default_rng(20260817)
BO_SEEDS = [int(RNG.integers(0, 2**31 - 1)) for _ in range(K_BO_SEEDS)]

OUTPUT_JSON = REPO_ROOT / "outputs" / "damped_sw_alpha_bo.json"

rust_fn = getattr(rust_core, "evaluate_alpha_tune_outcomes_py", None) if rust_core else None
print(f"policy: {POLICY}")
print(f"Rust kernel: {rust_available() and rust_fn is not None}")
print(f"Modal: {USE_MODAL}")
print(f"α bounds: {ALPHA_BOUNDS}")
print(f"episode: n_burn={N_BURN}, n_score={N_SCORE}")
print(f"target Ax trials={TOTAL_AX_TRIALS} in {N_BATCHES} batches of {TRIALS_PER_BATCH}, batch parallelism={AX_PARALLELISM}")
print(f"shard cap: modal={MODAL_CONCURRENCY}")
print(f"BO seeds (K={K_BO_SEEDS}): {BO_SEEDS}")

%matplotlib inline
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "grid.alpha": 0.3})


## Constant parameters

Edit the scalars below before running the objective / BO cells. These feed `ProfitCosts`, `ModelParams`, shipments, and `lead_time` on the evaluation path.

### Calendar demand (`USE_CALENDAR_DEMAND`)

When **True**, `MODEL_PARAMS` carries the committed FreshNet DOW×week profile (`data/freshnet/demand_profile.json`). Both **realized demand** in the episode and the SW protection target `d_star = protection_demand_quantile(α, …, start_day=order_day)` use calendar μ(day).

- **Flat μ path** (`USE_CALENDAR_DEMAND = False`): legacy homogeneous NB over the protection window (length still 3/3/4 on Sun/Tue/Thu).
- **Calendar path**: heterogeneous MC sum (n_mc=20_000, ADR 0134) over the days in each protection window.

**Rust note:** when `_core` is built, `alpha_tune` stays **Rust-first** with calendar demand via typed `_core.DemandProfile` (T-133). Without `_core`, evaluation falls back to Python.


In [ ]:
# --- Episode economics (SIM-01=B; ADR 0104 scaffold — uncalibrated) ---
UNIT_MARGIN = 2.0
WASTE_COST = 5.0
STOCKOUT_PENALTY = 3.0

costs = ProfitCosts(
    unit_margin=UNIT_MARGIN,
    waste_cost=WASTE_COST,
    stockout_penalty=STOCKOUT_PENALTY,
)

# --- Shipments ---
USE_ABDELLA = False  # True → `default_shipments()`; needs `uv sync --extra data`

# --- Demand / spoilage / ordering ---
DEMAND_MU = 30.0
DEMAND_VM = 2.0  # variance/mean; must be > 1
CASE_SIZE = 8
LEAD_TIME = 1  # days (passed to Rust kernel when _core is available)

# CAL-B4: FreshNet calendar profile (ADR 0113 / T-132 MC protection quantile)
USE_CALENDAR_DEMAND = True
DEMAND_PROFILE_PATH = REPO_ROOT / "data" / "freshnet" / "demand_profile.json"

# SW damping (Nahmias); tuned jointly with alpha for damped_sw
# 2026-08-30: widened 0.5-1.0 -> 0.5-2.0 per Oliver (also updated on the Studio
# rho slider, web/src/controls.ts) — see .team/plans/2026-08-29-gsin-belief-accuracy-blog-post.md §2.1
RHO_BOUNDS = (0.5, 2.0)
DEFAULT_RHO = 0.8

_demand_profile = (
    load_demand_profile(DEMAND_PROFILE_PATH) if USE_CALENDAR_DEMAND else None
)
MODEL_PARAMS = ModelParams(
    demand_mu=DEMAND_MU,
    demand_vm=DEMAND_VM,
    case_size=CASE_SIZE,
    demand_profile=_demand_profile,
)

shipments = default_shipments() if USE_ABDELLA else smoke_cool_shipments()

_eval_backend = (
    "rust"
    if rust_available() and rust_fn is not None
    else "python"
)

print(f"costs: margin={UNIT_MARGIN}, waste={WASTE_COST}, stockout={STOCKOUT_PENALTY}")
print(f"shipments: {'Abdella' if USE_ABDELLA else 'smoke-cool'} ({len(shipments)} trace(s))")
print(f"model: μ={DEMAND_MU}, V/M={DEMAND_VM}, case={CASE_SIZE}, lead_time={LEAD_TIME}")
print(f"calendar demand: {USE_CALENDAR_DEMAND} → eval backend: {_eval_backend}")
print(f"α bounds: {ALPHA_BOUNDS}; ρ bounds: {RHO_BOUNDS}")

SOO_BUDGETS = DampedSwSooBudgets(
    n_burn=N_BURN,
    n_score=N_SCORE,
    lead_time=LEAD_TIME,
    unit_margin=UNIT_MARGIN,
    waste_cost=WASTE_COST,
    stockout_penalty=STOCKOUT_PENALTY,
    demand_mu=DEMAND_MU,
    demand_vm=DEMAND_VM,
    case_size=CASE_SIZE,
    use_calendar_demand=USE_CALENDAR_DEMAND,
    demand_profile_path=_MODAL_DEMAND_PROFILE if USE_MODAL else str(DEMAND_PROFILE_PATH),
    use_abdella=USE_ABDELLA,
)


## Calendar protection targets (diagnostic)

Compare flat-μ vs calendar MC protection quantiles `d_star` on MWF order days. This is the quantity inside `q = case_round(ρ · max(0, d_star − Ĩ))`.


In [ ]:
ALPHA_DIAG = 0.9
flat_params = ModelParams(demand_mu=DEMAND_MU, demand_vm=DEMAND_VM)
schedule = DEFAULT_ORDER_SCHEDULE
order_days = [(d, schedule.protection_days(d)) for d in range(14) if schedule.can_order(d)]

rows = []
for day, prot in order_days[:3]:
    d_flat = protection_demand_quantile(
        ALPHA_DIAG, flat_params, protection_days=prot, start_day=day
    )
    d_cal = protection_demand_quantile(
        ALPHA_DIAG, MODEL_PARAMS, protection_days=prot, start_day=day
    )
    mus = [MODEL_PARAMS.demand_mu_for_day(day + k) for k in range(prot)]
    rows.append((day, prot, float(np.mean(mus)), d_flat, d_cal, d_cal - d_flat))

print(f"α={ALPHA_DIAG}: flat μ={DEMAND_MU} vs calendar profile")
for day, prot, mu_bar, d_flat, d_cal, delta in rows:
    print(
        f"  day={day:2d} prot={prot}  mean μ≈{mu_bar:5.1f}  "
        f"d*_flat={d_flat:6.1f}  d*_cal={d_cal:6.1f}  Δ={delta:+6.1f}"
    )


## Objective: episode outcomes with demand replicates

One Ax observation per (α, ρ) tuple = mean and SEM of K episode runs at fixed `BO_SEEDS`.


In [ ]:
def _episode_kwargs() -> dict[str, object]:
    return {
        "params": MODEL_PARAMS,
        "shipments": shipments,
        "costs": costs,
        "n_burn": N_BURN,
        "n_score": N_SCORE,
        "lead_time": LEAD_TIME,
    }


def evaluate_arm_outcomes(alpha: float, rho: float, root_seed: int):
    return evaluate_alpha_episode_outcomes(
        TUNE_ARM,
        float(alpha),
        int(root_seed),
        rho=float(rho),
        **_episode_kwargs(),
    )


def _replicate_mean_sem(values: list[float]) -> tuple[float, float]:
    arr = np.asarray(values, dtype=float)
    mean = float(arr.mean())
    sem = float(arr.std(ddof=1) / np.sqrt(len(arr))) if len(arr) > 1 else 0.0
    return mean, sem


def evaluate_with_replicates(
    alpha: float,
    rho: float,
    seeds: list[int],
) -> dict[str, tuple[float, float]]:
    """Mean and SEM over K demand seeds for profit / waste / stockout metrics."""
    profits: list[float] = []
    wastes: list[float] = []
    stockouts: list[float] = []
    for seed in seeds:
        out = evaluate_arm_outcomes(alpha, rho, seed)
        profits.append(out.profit)
        wastes.append(float(out.total_waste))
        stockouts.append(float(out.total_lost_sales))
    p_mean, p_sem = _replicate_mean_sem(profits)
    w_mean, w_sem = _replicate_mean_sem(wastes)
    s_mean, s_sem = _replicate_mean_sem(stockouts)
    return {
        "episode_profit": (p_mean, p_sem),
        "total_waste": (w_mean, w_sem),
        "total_stockout": (s_mean, s_sem),
    }


def ax_parameter_configs() -> list[RangeParameterConfig]:
    return [
        RangeParameterConfig(
            name="alpha",
            parameter_type="float",
            bounds=ALPHA_BOUNDS,
        ),
        RangeParameterConfig(
            name="rho",
            parameter_type="float",
            bounds=RHO_BOUNDS,
        ),
    ]


demo = evaluate_with_replicates(0.9, DEFAULT_RHO, BO_SEEDS[:2])
print(
    f"smoke {TUNE_ARM} α=0.9 ρ={DEFAULT_RHO} on 2 seeds: "
    f"profit={demo['episode_profit'][0]:.2f}±{demo['episode_profit'][1]:.3f}, "
    f"waste={demo['total_waste'][0]:.1f}, stockout={demo['total_stockout'][0]:.1f}"
)

def plot_bo_run(
    client: Client,
    trial_log: list[dict[str, Any]],
    *,
    title: str,
    best_params: dict[str, float],
    objective_key: str = "mean_profit",
) -> None:
    """Matplotlib summary for a 2D (alpha, rho) BO run."""
    alphas = np.array([t["alpha"] for t in trial_log])
    rhos = np.array([t["rho"] for t in trial_log])
    profits = np.array([t[objective_key] for t in trial_log])
    sems = np.array([t["sem_profit"] for t in trial_log])
    trials = np.array([t["trial_index"] for t in trial_log])
    best_so_far = np.maximum.accumulate(profits)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    sc = axes[0].scatter(alphas, rhos, c=profits, cmap="viridis", s=55, alpha=0.85)
    axes[0].scatter(
        [best_params["alpha"]],
        [best_params["rho"]],
        marker="*",
        s=220,
        c="#16a34a",
        label="best",
        zorder=5,
    )
    axes[0].set_xlabel("α")
    axes[0].set_ylabel("ρ")
    axes[0].set_title(f"{title}: trial locations")
    fig.colorbar(sc, ax=axes[0], label="replicate mean profit")
    axes[0].legend(fontsize=8)

    axes[1].errorbar(alphas, profits, yerr=sems, fmt="o", alpha=0.75, capsize=3)
    axes[1].set_xlabel("α")
    axes[1].set_ylabel("Episode profit (replicate mean)")
    axes[1].set_title("Profit vs α")

    axes[2].plot(trials, profits, "o", alpha=0.5, label="trial mean")
    axes[2].plot(trials, best_so_far, "-", color="#16a34a", lw=2, label="best-so-far")
    axes[2].set_xlabel("Ax trial index")
    axes[2].set_ylabel("Replicate mean profit")
    axes[2].set_title("Convergence")
    axes[2].legend(fontsize=8)

    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    plt.show()


## Textbook fractile reference (CTL-03 context)

In [ ]:
alpha_theory_penalty = costs.stockout_penalty / (
    costs.stockout_penalty + costs.waste_cost
)
alpha_theory_margin = costs.unit_margin / (costs.unit_margin + costs.waste_cost)
print(f"Textbook (penalty / waste): {alpha_theory_penalty:.3f}")
print(f"Textbook (margin / waste):  {alpha_theory_margin:.3f}")

## Modal worker (inline app + wheel)

Build the Rust wheel before the first Modal run:

```bash
mkdir -p dist/wheel
uv run maturin build --release --manifest-path crates/voi_py/Cargo.toml -o dist/wheel
```

Flat pool: all `(trial × seed)` shards in one batch spawn+get loop (cap `MODAL_CONCURRENCY`).

In [ ]:
MODAL_FUNCTION_TIMEOUT_S = 600.0
modal_soo_shard = None

if USE_MODAL:
    try:
        import modal
    except ImportError as exc:
        raise SystemExit("Install modal: uv sync --extra modal") from exc

    _PKG_SRC = REPO_ROOT / "src" / "blueberries_voi"
    _DATA_DIR = REPO_ROOT / "data"
    _WHEEL_DIR = Path(os.environ.get("BLUEBERRIES_VOI_WHEEL", REPO_ROOT / "dist" / "wheel"))
    if _WHEEL_DIR.is_dir():
        _wheel_files = sorted(_WHEEL_DIR.glob("blueberries_voi_core-*.whl"))
        WHEEL_PATH = (
            _wheel_files[-1]
            if _wheel_files
            else (_WHEEL_DIR / "blueberries_voi_core.whl")
        )
    else:
        WHEEL_PATH = _WHEEL_DIR
    _WHEEL_REMOTE = f"/tmp/{WHEEL_PATH.name}"

    _UV_PROJECT_DIR = os.path.relpath(REPO_ROOT, Path.cwd().resolve())
    if _UV_PROJECT_DIR == os.curdir:
        _UV_PROJECT_DIR = "."

    _thread_env = {
        "PYTHONPATH": "/root",
        "BLUEBERRIES_VOI_BACKEND": "rust",
        "OMP_NUM_THREADS": "1",
        "MKL_NUM_THREADS": "1",
        "OPENBLAS_NUM_THREADS": "1",
    }
    _base_image = modal.Image.debian_slim(python_version="3.11").uv_sync(
        uv_project_dir=_UV_PROJECT_DIR,
        extras=["data"],
        frozen=True,
    )
    if modal.is_local():
        image = _base_image.add_local_dir(
            str(_PKG_SRC), remote_path="/root/blueberries_voi", copy=True
        )
        image = image.add_local_dir(str(_DATA_DIR), remote_path="/data", copy=True)
        if WHEEL_PATH.is_file():
            image = image.add_local_file(str(WHEEL_PATH), _WHEEL_REMOTE, copy=True)
            image = image.run_commands(f"pip install {_WHEEL_REMOTE}")
        image = image.env(_thread_env)
    else:
        image = _base_image.env(_thread_env)

    app = modal.App("blueberries-voi-damped-sw-soo", image=image)

    @app.function(timeout=int(MODAL_FUNCTION_TIMEOUT_S), cpu=1.0)
    def soo_shard(job: dict[str, Any]) -> dict[str, Any]:
        return run_soo_shard(job)

    modal_soo_shard = soo_shard
    print(f"Modal wheel: {WHEEL_PATH}")


def _evaluate_batch_modal(jobs: list[dict[str, Any]]) -> list[dict[str, Any]]:
    assert modal_soo_shard is not None
    if not jobs:
        return []

    shards: list[dict[str, Any]] = []
    with app.run():
        for chunk_start in range(0, len(jobs), MODAL_CONCURRENCY):
            chunk = jobs[chunk_start : chunk_start + MODAL_CONCURRENCY]
            handles = [(job, modal_soo_shard.spawn(job)) for job in chunk]
            with ThreadPoolExecutor(max_workers=len(handles)) as pool:
                futs = {
                    pool.submit(
                        h.get,
                        timeout=MODAL_FUNCTION_TIMEOUT_S + 10.0,
                    ): job
                    for job, h in handles
                }
                for fut in as_completed(futs):
                    shards.append(fut.result())
    return shards


def _evaluate_batch_local(jobs: list[dict[str, Any]]) -> list[dict[str, Any]]:
    from concurrent.futures import ProcessPoolExecutor

    if not jobs:
        return []
    max_workers = min(len(jobs), os.cpu_count() or 2)
    shards: list[dict[str, Any]] = []
    with ProcessPoolExecutor(max_workers=max_workers) as pool:
        futs = {pool.submit(run_soo_shard, job): job for job in jobs}
        for fut in as_completed(futs):
            shards.append(fut.result())
    return shards


def evaluate_ax_batch(
    trials: dict[int, dict[str, object]],
    seeds: list[int],
) -> dict[int, dict[str, tuple[float, float]]]:
    jobs = build_soo_jobs(trials, seeds, SOO_BUDGETS)
    if USE_MODAL:
        shards = _evaluate_batch_modal(jobs)
    else:
        shards = _evaluate_batch_local(jobs)
    return aggregate_soo_shards(shards)

## Run 1 — single-objective profit maximization (Modal + Ax)

Joint Ax search over **(α, ρ)** maximizing `episode_profit`, split into
**4 cells of `TRIALS_PER_BATCH` (12) trials each** — run them one at a time;
each prints the best `(α, ρ)` found so far and the improvement over the
previous batch, so it's easy to eyeball convergence and stop early if the
last couple of batches aren't finding anything better. The 4 batch cells
below are (almost) identical — same one-line call to `run_ax_batch`, only
the batch number differs — by design, so stopping partway through just means
skipping the remaining cells.

- Request **`AX_PARALLELISM`** candidates per round within a batch.
- Evaluate all `trial × seed` shards in one flat Modal pool per round.
- **`client.save_to_json_file`** after every round (not just every batch), so
  the Ax client on disk is always current even if you stop mid-batch.


In [ ]:
def _completed_trial_count(ax_client: Client) -> int:
    return sum(1 for t in ax_client._experiment.trials.values() if t.status.is_completed)


def run_ax_batch(batch_num: int, n_trials_this_batch: int, total_batches: int = N_BATCHES):
    """Run one batch of `n_trials_this_batch` Ax trials. Batch 1 creates a fresh
    client (or reloads one from AX_JSON if RELOAD_AX=True); batches 2+ continue
    the client left in scope by the previous batch. Prints best-so-far and the
    improvement vs. the previous batch so a manual run is easy to eyeball and
    stop early. Returns (client, trial_log, best_alpha, best_rho, best_profit)."""
    global client_profit, trial_log_profit

    prev_best = max((t["mean_profit"] for t in trial_log_profit), default=None) if batch_num > 1 else None

    if batch_num == 1:
        if RELOAD_AX and AX_JSON.is_file():
            client_profit = Client.load_from_json_file(str(AX_JSON))
            completed_existing = _completed_trial_count(client_profit)
            trial_log_profit = []
            print(f"Loaded Ax client from {AX_JSON.name} ({completed_existing} trials already completed)")
        else:
            client_profit = Client()
            client_profit.configure_experiment(
                name="damped-sw-soo-modal",
                parameters=ax_parameter_configs(),
            )
            client_profit.configure_optimization(objective="episode_profit")
            trial_log_profit = []

    completed = 0
    pbar = tqdm(total=n_trials_this_batch, desc=f"Ax batch {batch_num}/{total_batches}")
    while completed < n_trials_this_batch:
        batch_n = min(AX_PARALLELISM, n_trials_this_batch - completed)
        trials = client_profit.get_next_trials(max_trials=batch_n)
        metrics_by_trial = evaluate_ax_batch(trials, BO_SEEDS)
        for trial_index, parameters in trials.items():
            metrics = metrics_by_trial[int(trial_index)]
            client_profit.complete_trial(
                trial_index=trial_index,
                raw_data={"episode_profit": metrics["episode_profit"]},
            )
            trial_log_profit.append(
                {
                    "trial_index": int(trial_index),
                    "alpha": float(parameters["alpha"]),
                    "rho": float(parameters["rho"]),
                    "mean_profit": metrics["episode_profit"][0],
                    "sem_profit": metrics["episode_profit"][1],
                    "mean_waste": metrics["total_waste"][0],
                    "mean_stockout": metrics["total_stockout"][0],
                }
            )
        AX_JSON.parent.mkdir(parents=True, exist_ok=True)
        client_profit.save_to_json_file(str(AX_JSON))
        completed += len(trials)
        pbar.update(len(trials))
    pbar.close()

    best_params, _pred, best_index, _name = client_profit.get_best_parameterization()
    best_alpha = float(best_params["alpha"])
    best_rho = float(best_params["rho"])
    best_profit = max(t["mean_profit"] for t in trial_log_profit)
    total_completed = len(trial_log_profit)

    print(f"=== Batch {batch_num}/{total_batches} done: {total_completed} trials total (this batch: {n_trials_this_batch}) ===")
    print(f"best so far: alpha={best_alpha:.4f}, rho={best_rho:.4f}, profit={best_profit:.2f} (trial {best_index})")
    if prev_best is not None:
        delta = best_profit - prev_best
        verdict = "improved" if delta > 1e-9 else "no improvement (may be converged, or just noise)"
        print(f"vs. previous batch: {delta:+.2f} ({verdict})")
    print(f"Saved Ax client -> {AX_JSON}")

    return client_profit, trial_log_profit, best_alpha, best_rho, best_profit


In [ ]:
# Batch 1 of 4 — fresh Ax client (or reloaded, if RELOAD_AX=True)
client_profit, trial_log_profit, best_alpha_profit, best_rho_profit, _best_profit = \
    run_ax_batch(1, TRIALS_PER_BATCH)


In [ ]:
# Batch 2 of 4 — continues the client from batch 1
client_profit, trial_log_profit, best_alpha_profit, best_rho_profit, _best_profit = \
    run_ax_batch(2, TRIALS_PER_BATCH)


In [ ]:
# Batch 3 of 4 — continues the client from batch 2
client_profit, trial_log_profit, best_alpha_profit, best_rho_profit, _best_profit = \
    run_ax_batch(3, TRIALS_PER_BATCH)


In [ ]:
# Batch 4 of 4 — continues the client from batch 3
client_profit, trial_log_profit, best_alpha_profit, best_rho_profit, _best_profit = \
    run_ax_batch(4, TRIALS_PER_BATCH)


## Run 1 diagnostics (profit SOO)

In [ ]:
from ax.analysis.plotly import SlicePlot

if len(trial_log_profit) >= 8:
    client_profit.compute_analyses(analyses=[SlicePlot(parameter_name="alpha")])
    client_profit.compute_analyses(analyses=[SlicePlot(parameter_name="rho")])

plot_bo_run(
    client_profit,
    trial_log_profit,
    title=f"Run 1 — profit SOO ({TUNE_ARM})",
    best_params={"alpha": best_alpha_profit, "rho": best_rho_profit},
)


## Grid baseline and validation (removed)

The grid baseline loop (`DEFAULT_DESKTOP_ALPHAS`), `tune_alpha_grid` CRN pass, and held-out `validation_mean` local evaluation were removed from this notebook. Use Modal Ax BO only (`evaluate_ax_batch` / `scripts/run_damped_sw_soo_bo.py`). Reload results from `outputs/damped_sw_alpha_bo.json` if you need the best (α, ρ) without re-running trials.

In [ ]:
# Grid baseline and tune_alpha_grid CRN pass removed — see section above.


In [ ]:
# Held-out validation_mean local pass removed — Ax replicate mean is the objective.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.errorbar(
    [t["alpha"] for t in trial_log_profit],
    [t["mean_profit"] for t in trial_log_profit],
    yerr=[t["sem_profit"] for t in trial_log_profit],
    fmt="o",
    alpha=0.7,
    label="Ax trials",
)
ax.axvline(best_alpha_profit, color="#16a34a", ls="--", label=f"Ax α*={best_alpha_profit:.2f}")
ax.axvline(alpha_theory_penalty, color="#dc2626", ls=":", alpha=0.7, label="theory (penalty)")
ax.set_xlabel("α")
ax.set_ylabel("Episode profit (replicate mean)")
ax.set_title(f"Ax BO — {TUNE_ARM} (ρ={best_rho_profit:.2f})")
ax.legend(fontsize=8, loc="best")
fig.tight_layout()
plt.show()


## Save results (optional)

Writes to `outputs/` (gitignored) — not `experiments/tuned_alpha.json`.

In [ ]:
payload: dict[str, Any] = {
    "policy": POLICY,
    "full_run": FULL_RUN,
    "use_modal": USE_MODAL,
    "modal_concurrency": MODAL_CONCURRENCY,
    "ax_parallelism": AX_PARALLELISM,
    "total_ax_trials": TOTAL_AX_TRIALS,
    "ax_client_path": str(AX_JSON.relative_to(REPO_ROOT)),
    "rust_kernel": bool(rust_available() and rust_fn is not None),
    "alpha_bounds": list(ALPHA_BOUNDS),
    "rho_bounds": list(RHO_BOUNDS),
    "n_burn": N_BURN,
    "n_score": N_SCORE,
    "bo_seeds": BO_SEEDS,
    "costs": {
        "unit_margin": UNIT_MARGIN,
        "waste_cost": WASTE_COST,
        "stockout_penalty": STOCKOUT_PENALTY,
    },
    "model_params": {
        "demand_mu": DEMAND_MU,
        "demand_vm": DEMAND_VM,
        "use_calendar_demand": USE_CALENDAR_DEMAND,
        "case_size": CASE_SIZE,
        "lead_time": LEAD_TIME,
        "use_abdella": USE_ABDELLA,
    },
    "best_alpha_profit_soo": best_alpha_profit,
    "best_rho_profit_soo": best_rho_profit,
    "trials_profit_soo": trial_log_profit,
}
OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
print(f"Wrote {OUTPUT_JSON}")


## Takeaways

1. **damped_sw** = survival-weighted base-stock: `q = case_round(ρ · max(0, F⁻¹(α) − Ĩ))`.
2. Ax receives `(mean, sem)` per metric over K demand seeds.
3. Rebuild `maturin develop` after pulling ρ-aware `voi_core` changes.
4. With calendar demand, use `FULL_RUN = True` so burn-in covers MWF cadence.
